In [107]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## Questão 01 

a) Considerando uma validação cruzada em 10 folds, avalie modelos de *classificação binária* nos dados em questão.

In [108]:
# extraindo dados
breast = np.genfromtxt("breastcancer.csv", delimiter=',')

In [109]:
np.shape(breast)

(569, 31)

In [110]:
# função que divide o dataset em k-folds

def kfold_ind(k, X, y):
    tam_fold = len(X) // k
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    folds_norm = []
    
    for i in range(k):
        teste_indices = indices[i * tam_fold: (i+1) * tam_fold]
        treino_indices = np.concatenate([indices[:i * tam_fold], indices[(i+1) * tam_fold:]])
        
        X_treino, X_teste = X[treino_indices], X[teste_indices]
        y_treino, y_teste = y[treino_indices], y[teste_indices]
        
        media = np.mean(X_treino, axis=0)
        desvio = np.std(X_treino, axis=0)
        
        X_treino_norm = (X_treino - media) / desvio
        X_teste_norm = (X_teste - media) / desvio
        
        X_treino_norm = np.concatenate((np.ones((X_treino_norm.shape[0], 1)), X_treino_norm), axis=1)
        X_teste_norm = np.concatenate((np.ones((X_teste_norm.shape[0], 1)), X_teste_norm), axis=1)
        
        folds_norm.append({
            'X_treino': X_treino_norm,
            'y_treino': y_treino,
            'X_teste': X_teste_norm,
            'y_teste': y_teste,
            'indices_treino': treino_indices,
            'indices_teste': teste_indices
        })
    
    return folds_norm
        

### Regressão logística

In [111]:
# usando GD
def gd_logistic_regression(alfa, epsilon, epocas, X, y, w):
    n = X.shape[0]
    custo = []
    for epoca in range(epocas):
        z = X @ w
        
        y_est = 1 / (1 + np.exp(-z))
        ei = y - y_est
        w = w + alfa * (X.T @ ei) / n
        
        f_custo = -np.mean(y * np.log(y_est + epsilon) + (1 - y) * np.log(1 - y_est + epsilon))
        custo.append(f_custo)
    
    return w, custo

In [112]:
y = breast[:, [30]] # y não precisa ser normalizado, pois já está em termos categóricos!
X_raw = breast[:, 0:30] # vetor X sem o intercepto

folds = kfold_ind(10, X_raw, y)

pesos_finais = []
for fold_idx, fold in enumerate(folds):
    X_treino_fold = fold['X_treino']
    y_treino_fold = fold['y_treino']
    
    w = np.ones((X_treino_fold.shape[1], 1))
    
    w_final, historico_custo = gd_logistic_regression(0.001, 1e-7, 1000, X_treino_fold, y_treino_fold, w)
    pesos_finais.append(w_final)
    print(f"Fold {fold_idx + 1} -> Custo final: {historico_custo[-1]}")

Fold 1 -> Custo final: 0.687299241861514
Fold 2 -> Custo final: 0.6979200435972787
Fold 3 -> Custo final: 0.7209480081012875
Fold 4 -> Custo final: 0.6600892388318038
Fold 5 -> Custo final: 0.649775086532242
Fold 6 -> Custo final: 0.6666116967487757
Fold 7 -> Custo final: 0.6701643291597025
Fold 8 -> Custo final: 0.7456847986168892
Fold 9 -> Custo final: 0.7608811859400303
Fold 10 -> Custo final: 0.75148414974093


### Análise do discriminante Gaussiano

### Naive Bayes Gaussiano